# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring  
**Model being audited:** the five-feature Logistic Regression from Week 5.

This notebook applies the same skepticism to my own work that the FlyRank research paper applies to public claims. I first ask methodology questions about two paper findings, then compare a convenient random-row split with the more deployment-relevant **client-grouped split**, audit leakage, inspect failures, and rewrite my own claim to match the evidence.

All results below use the real 30,000-row starter dataset. Language is intentionally public-safe: **observed, measured, directional, and decision-support** rather than causal.


## 1. Two paper findings + my methodology questions

### Paper finding 1 — “The Anatomy of Growing Content”

The paper reports that the growing cohort is **37.6% longer** (about 3.2K vs 2.3K words) and **20% younger** (184 vs 230 days) than the declining cohort. It describes this as an observational comparison and notes large sample sizes.

**Methodology question I would ask:** The up/down grouping comes from a 30-day-vs-previous-30-day impression trend. Are word count, age, content type, intent, and client/site differences all measured or controlled in a way that prevents the cohort comparison from mixing structural differences with the trend label? A client-grouped or matched analysis would help show whether the measured gap persists on unseen brands rather than being driven by portfolio composition.

**Respectful reading:** The finding supports a **directional association** in this portfolio. I would avoid reading “longer” or “younger” as a causal treatment without a design that separates those factors from client, intent, topic, and selection effects.

### Paper finding 2 — “The Age-Freshness Matrix”

The paper reports health around **44.62** for old (365+) but recently refreshed pages and **44.12** for young-fresh pages, and presents this as evidence that old content can perform strongly after updates. The paper also explicitly warns about survivor bias in a small stale-old cell.

**Methodology question I would ask:** How was “refreshed” assigned, and were refreshed old pages systematically stronger or more commercially important *before* editors chose to update them? A matched before/after or time-aware comparison of refreshed versus similar unrevised pages would better support a treatment claim. I would also ask how much the composite health score overlaps with the raw outcomes used to judge success.

**Respectful reading:** The matrix is useful evidence that refreshed old pages are **observed alongside strong measured performance**. Without random or quasi-experimental assignment, I would phrase it as association rather than proof that the refresh itself caused the result.


In [1]:
import pandas as pd

paper_checks = pd.DataFrame([
    {
        "paper_finding": "Growing vs declining content",
        "reported_measurement": "3.2K vs 2.3K words; 184d vs 230d age",
        "label_or_group_source": "30d vs previous-30d impression trend",
        "methodology_question": "Does the gap survive client/content-type controls or grouped validation?"
    },
    {
        "paper_finding": "Old + recently refreshed vs young + fresh",
        "reported_measurement": "Health 44.62 vs 44.12",
        "label_or_group_source": "Age tier × days-since-update tier",
        "methodology_question": "Were refreshed pages selected differently before refresh; is there a matched/time-aware comparison?"
    },
])

print("Two paper findings selected for constructive methodology review:")
display(paper_checks)


Two paper findings selected for constructive methodology review:


,paper_finding,reported_measurement,label_or_group_source,methodology_question
0,Growing vs declining content,3.2K vs 2.3K words; 184d vs 230d age,30d vs previous-30d impression trend,Does the gap survive client/content-type contr...
1,Old + recently refreshed vs young + fresh,Health 44.62 vs 44.12,Age tier × days-since-update tier,Were refreshed pages selected differently befo...


## 2. My model under an honest split (before/after)

My Week-5 model used five measured inputs:

`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `days_since_last_update`

and evaluated the observed proxy:

`decline_proxy = 1` when `trend_direction == "down"`.

For this audit I show a **before/after validation improvement**:

- **Before:** stratified random row split. This is convenient but allows pages from the same client to appear in both train and test.
- **After:** `GroupShuffleSplit` by `client_id`. Every held-out client is unseen during training.

The model and feature set are identical. Only the validation design changes. I report the base rate, Precision@20/50, AUROC, and Average Precision. Because Precision@20 uses only twenty rows, I read it beside the broader metrics rather than treating it as sufficient proof by itself.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]
X = df[features].copy()
y = df["decline_proxy"].astype(int)

def build_model():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def precision_at_k(y_true, scores, k):
    yt = np.asarray(y_true)
    order = np.argsort(-np.asarray(scores))[:k]
    return float(yt[order].mean())

def evaluate_split(name, train_idx, test_idx):
    model = build_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    yt = y.iloc[test_idx].to_numpy()

    result = {
        "validation": name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "base_rate": float(yt.mean()),
        "Precision@20": precision_at_k(yt, scores, 20),
        "Precision@50": precision_at_k(yt, scores, 50),
        "AUROC": roc_auc_score(yt, scores),
        "Average Precision": average_precision_score(yt, scores),
    }
    return result, model, scores

all_idx = np.arange(len(df))

# BEFORE: random rows; stratified only on the proxy.
random_train, random_test = train_test_split(
    all_idx,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
random_result, random_model, random_scores = evaluate_split(
    "Before: random rows", random_train, random_test
)

# AFTER: all rows from a client remain together.
group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train, group_test = next(
    group_split.split(X, y, groups=df["client_id"])
)
group_result, group_model, group_scores = evaluate_split(
    "After: grouped by client", group_train, group_test
)

train_clients = set(df.iloc[group_train]["client_id"])
test_clients = set(df.iloc[group_test]["client_id"])
assert train_clients.isdisjoint(test_clients)

comparison = pd.DataFrame([random_result, group_result])
print(f"Random split client overlap: "
      f"{len(set(df.iloc[random_train]['client_id']) & set(df.iloc[random_test]['client_id']))} clients")
print(f"Grouped split client overlap: {len(train_clients & test_clients)} clients")
print(f"Grouped train/test clients: {len(train_clients)} / {len(test_clients)}")

display(
    comparison.style.format({
        "base_rate": "{:.1%}",
        "Precision@20": "{:.1%}",
        "Precision@50": "{:.1%}",
        "AUROC": "{:.3f}",
        "Average Precision": "{:.3f}",
    })
)

print(
    "\nAudit reading: the grouped AUROC and Average Precision are the safer overall "
    "generalization read. Precision@20 is capacity-aligned but uses only 20 cases, "
    "so I do not use it alone to claim model skill."
)


Random split client overlap: 31 clients
Grouped split client overlap: 0 clients
Grouped train/test clients: 24 / 8


,validation,train_rows,test_rows,base_rate,Precision@20,Precision@50,AUROC,Average Precision
0,Before: random rows,22500,7500,54.2%,20.0%,36.0%,0.542,0.584
1,After: grouped by client,22885,7115,51.7%,75.0%,62.0%,0.501,0.525



Audit reading: the grouped AUROC and Average Precision are the safer overall generalization read. Precision@20 is capacity-aligned but uses only 20 cases, so I do not use it alone to claim model skill.


## 3. Leakage audit

I audit the final five-feature set against three leakage families.

**Label-derived leakage:** `decline_proxy` comes from `trend_direction`, which is computed from `trend_pct`. Therefore `trend_direction`, `trend_pct`, and the proxy itself are forbidden as features.

**Future/overlapping-window leakage:** the starter dataset is a single trailing-90-day snapshot, so it does **not** provide a clean past-feature / future-label timeline. That is a limitation of this Week-5 exercise. I therefore do not describe this as a production-ready future prediction model. A warehouse version should use feature windows strictly before a future outcome window.

**Decision-derived leakage:** IDs, existing product flags, recommendation/action columns, and previous model scores must not enter the feature matrix. `client_id` is used only to create the grouped split.

A second issue is validation leakage through repeated clients. The random row split exposes every client to both sides; the grouped split removes that path.


In [3]:
# Mechanical leakage audit of the actual Week-5 feature list.
label_family = {"decline_proxy", "trend_direction", "trend_pct", "is_declining_label"}
id_family = {"client_id", "content_id"}
decision_family = {
    "action_label", "reason_code", "baseline_action_score",
    "health_score", "optimization_flag", "model_score"
}

audit_rows = []
for f in features:
    audit_rows.append({
        "feature": f,
        "label_derived?": f in label_family,
        "ID?": f in id_family,
        "decision_derived?": f in decision_family,
        "missing_rows": int(df[f].isna().sum()),
        "kept?": True,
    })

audit = pd.DataFrame(audit_rows)
display(audit)

assert set(features).isdisjoint(label_family)
assert set(features).isdisjoint(id_family)
assert set(features).isdisjoint(decision_family)
assert len(train_clients & test_clients) == 0

print("Leakage audit PASSED for named feature families.")
print("Important limitation: the starter snapshot does not prove strict feature-before-label timing.")

# Real failure examples from the grouped holdout.
group_errors = df.iloc[group_test][[
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "days_since_last_update", "decline_proxy"
]].copy()
group_errors["model_score"] = group_scores

false_pos = (
    group_errors[group_errors["decline_proxy"] == 0]
    .sort_values("model_score", ascending=False)
    .head(5)
    .copy()
)

false_neg = (
    group_errors[group_errors["decline_proxy"] == 1]
    .sort_values("model_score", ascending=True)
    .head(5)
    .copy()
)

print("\nFive high-scoring false positives on unseen clients:")
display(false_pos)

print("\nFive low-scoring false negatives on unseen clients:")
display(false_neg)

print(
    "Failure interpretation: the same visibility/click/freshness measurements can occur "
    "for pages with different observed trend directions. Query intent, seasonality, SERP "
    "changes, content type, and client-specific context are plausible omitted factors."
)


,feature,label_derived?,ID?,decision_derived?,missing_rows,kept?
0,impressions_90d,False,False,False,0,True
1,clicks_90d,False,False,False,0,True
2,ctr,False,False,False,0,True
3,avg_position,False,False,False,0,True
4,days_since_last_update,False,False,False,0,True


Leakage audit PASSED for named feature families.
Important limitation: the starter snapshot does not prove strict feature-before-label timing.

Five high-scoring false positives on unseen clients:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,decline_proxy,model_score
9476,content_30eb41dff556,client_d029fa3a95,84,0,0.0,6.2,183,0,0.691140
20924,content_277eeb6d46cc,client_d029fa3a95,17,0,0.0,7.9,183,0,0.689368
12801,content_2265b3e09778,client_d029fa3a95,12,0,0.0,8.4,183,0,0.688852
28277,content_22ba8c872ab2,client_d029fa3a95,24,0,0.0,9.1,183,0,0.688137
5833,content_6557f2b648e8,client_d029fa3a95,18,0,0.0,9.2,183,0,0.688031



Five low-scoring false negatives on unseen clients:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,decline_proxy,model_score
21565,content_9532f197bbc8,client_4e07408562,309192,2689,0.87,2.0,104,1,0.012358
21819,content_4c36c775b818,client_4e07408562,463103,1889,0.41,2.3,20,1,0.052720
2365,content_4a18c07e5357,client_4e07408562,125049,1077,0.86,4.5,104,1,0.175404
9129,content_3db3c9053d5c,client_f369cb89fc,4,1,25.00,1.3,20,1,0.183958
17127,content_8818fd6d967f,client_4e07408562,83603,886,1.06,3.4,104,1,0.221516


Failure interpretation: the same visibility/click/freshness measurements can occur for pages with different observed trend directions. Query intent, seasonality, SERP changes, content type, and client-specific context are plausible omitted factors.


## 4. Claim rewrite

### Claim I would no longer make

> “The Week-5 model can identify declining pages and rank the best refresh opportunities.”

That sentence goes beyond the evidence. It mixes an observed trend proxy with a future-prediction claim and implies that a refresh is the correct treatment.

### Public-safe rewrite

**“On this starter snapshot, a five-feature Logistic Regression showed limited measured discrimination when evaluated on clients held out from training. Its grouped-holdout AUROC was close to chance overall, even though the top-20 queue happened to contain a high share of observed declining pages. I therefore treat the score as exploratory decision-support, not as evidence that it generalizes reliably to unseen clients or that refreshing a high-scoring page will cause recovery.”**

This rewrite matches the validation design. It reports what was measured, acknowledges the conflicting top-k versus global metrics, and avoids causal language.


In [4]:
# Turn the comparison into a concise evidence receipt used by the claim above.
before = comparison.loc[comparison["validation"] == "Before: random rows"].iloc[0]
after = comparison.loc[comparison["validation"] == "After: grouped by client"].iloc[0]

claim_receipt = pd.DataFrame([
    {
        "check": "AUROC",
        "before_random": before["AUROC"],
        "after_grouped": after["AUROC"],
        "interpretation": "Grouped value is the safer unseen-client generalization estimate."
    },
    {
        "check": "Average Precision",
        "before_random": before["Average Precision"],
        "after_grouped": after["Average Precision"],
        "interpretation": "Read beside each split's base rate."
    },
    {
        "check": "Precision@20",
        "before_random": before["Precision@20"],
        "after_grouped": after["Precision@20"],
        "interpretation": "Action-aligned but only 20 cases; useful, not sufficient alone."
    },
])

display(
    claim_receipt.style.format({
        "before_random": "{:.3f}",
        "after_grouped": "{:.3f}",
    })
)

print(
    f"Grouped holdout: AUROC={after['AUROC']:.3f}, "
    f"AP={after['Average Precision']:.3f} with base rate={after['base_rate']:.1%}, "
    f"Precision@20={after['Precision@20']:.1%}."
)
print("Final wording remains observational and decision-support only.")


,check,before_random,after_grouped,interpretation
0,AUROC,0.542,0.501,Grouped value is the safer unseen-client generalization estimate.
1,Average Precision,0.584,0.525,Read beside each split's base rate.
2,Precision@20,0.200,0.750,"Action-aligned but only 20 cases; useful, not sufficient alone."


Grouped holdout: AUROC=0.501, AP=0.525 with base rate=51.7%, Precision@20=75.0%.
Final wording remains observational and decision-support only.


## Self-check

- [x] Two concrete findings from the FlyRank research paper are named.
- [x] Each paper finding has a constructive methodology question about label/group definition, selection, or validation.
- [x] My Week-5 Logistic Regression is re-run under both a random-row split and a client-grouped split.
- [x] The before/after table uses the same model and features; only validation design changes.
- [x] Grouped validation has zero client overlap.
- [x] Base rate is printed beside Precision@20/50, AUROC, and Average Precision.
- [x] The final feature set is checked for label-derived, ID, and decision-derived leakage.
- [x] The starter snapshot's feature/label timing limitation is disclosed.
- [x] Real false-positive and false-negative examples from unseen clients are shown.
- [x] My strongest claim is rewritten using measured, observed, directional, and decision-support language.
- [x] I do not claim that a score proves a refresh will cause recovery.
- [x] The notebook runs top to bottom with no errors.
- [ ] Commit this executed notebook to `work/notebooks/w06_validation_audit.ipynb`, then submit the public repo URL.
